In [1]:
#Automated Analysis of Course Evaluation Comments-Hsu Nyein San

import numpy as np
import pandas as pd

In [2]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [3]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

In [4]:
data_set = pd.read_csv("https://drive.google.com/uc?id=1d3Fvaj91jvjzjDvf_eflvX-nzdboAwQy")
print(data_set.columns)

Index(['student_id', 'department', 'subject_name', 'feedback_text',
       'preprocessed_text', 'sentiment_label', 'sarcasm_flag', 'emotion_tag',
       'feedback_type', 'subject_specific_context', 'response_required'],
      dtype='object')


In [5]:
import re
from nltk.tokenize import word_tokenize

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    return " ".join(tokens)

data_set['clean_text'] = data_set['feedback_text'].apply(preprocess)

In [6]:
from sklearn.preprocessing import LabelEncoder

sentiment_le = LabelEncoder()
y = sentiment_le.fit_transform(data_set['sentiment_label'])

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)
X_text = vectorizer.fit_transform(data_set['clean_text'])

In [8]:
from scipy.sparse import hstack

if 'sarcasm_flag' in data_set.columns:
    X_sarcasm = data_set[['sarcasm_flag']].values
else:
    X_sarcasm = np.zeros((X_text.shape[0],1))

X = hstack([X_text, X_sarcasm])


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
from sklearn.linear_model import LogisticRegression

sentiment_model = LogisticRegression(solver='liblinear',class_weight='balanced',random_state=42)

sentiment_model.fit(X_train, y_train)


LogisticRegression(class_weight='balanced', random_state=42, solver='liblinear')

In [11]:
from textblob import TextBlob

def detect_sarcasm(text):
    if not text:
        return 0

    text_lower = text.lower()
    blob = TextBlob(text_lower)
    polarity = blob.sentiment.polarity

    sarcastic_keywords = ['great', 'fantastic', 'amazing', 'wonderful', 'awesome','perfect', 'excellent', 'yeah right', 'just what i needed']

    negative_context = ['boring', 'bad', 'slow', 'waste', 'useless', 'sleepy', 'die','annoying', 'terrible', 'horrible', 'too long','yet another', 'all weekend', 'never works', 'pop quiz']


    has_positive_kw = any(kw in text_lower for kw in sarcastic_keywords)
    has_negative_kw = any(nw in text_lower for nw in negative_context)

    if has_positive_kw and has_negative_kw:
        return 1

    if polarity > 0.1:
        if "yeah right" in text_lower or "thanks a lot" in text_lower:
            return 1

    return 0

In [12]:
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.8 MB/s eta 0:00:00


In [38]:
prediction_history = data_set[['department', 'subject_name', 'sentiment_label']].rename(
    columns={
        'department': 'Department',
        'subject_name': 'Subject',
        'sentiment_label': 'Sentiment',
        'sarcasm_flag': 'sarcasm_flag'
    }
).to_dict('records')



prediction_history = data_set[['department', 'subject_name', 'sentiment_label']].rename(
    columns={
        'department': 'Department',
        'subject_name': 'Subject',
        'sentiment_label': 'Sentiment'
    }
).to_dict('records')


def get_course_ranking(history_list, group_by_key="Subject"):
    try:
        if not history_list:
            return pd.DataFrame({"အသိပေးချက်": ["ဒေတာ မရှိသေးပါ။ Prediction အရင်လုပ်ပါ။"]})

        df = pd.DataFrame(list(history_list))
        df['Sentiment'] = df['Sentiment'].astype(str).str.strip().str.upper()


        ranking = df.groupby(group_by_key)['Sentiment'].value_counts().unstack(fill_value=0)


        for col in ['POSITIVE', 'NEUTRAL', 'NEGATIVE']:
            if col not in ranking.columns:
                ranking[col] = 0


        ranking['Total'] = ranking[['POSITIVE', 'NEUTRAL', 'NEGATIVE']].sum(axis=1)

        ranking['Quality_Score_%'] = ((ranking['POSITIVE'] + (ranking['NEUTRAL'] * 0.5)) / ranking['Total']) * 100
        ranking['Quality_Score_%'] = ranking['Quality_Score_%'].round(2)


        ranking = ranking.sort_values(by='Quality_Score_%', ascending=False).reset_index()
        ranking.insert(0, 'Rank', range(1, len(ranking) + 1))


        final_df = ranking.rename(columns={
            group_by_key: f"{group_by_key} Name",
            'POSITIVE': 'အကောင်းမြင် (Pos)',
            'NEUTRAL': 'ကြားနေ (Neu)',
            'NEGATIVE': 'အဆိုးမြင် (Neg)',
            'Total': 'စုစုပေါင်း',
            'Quality_Score_%': 'အရည်အသွေး (%)',
        })

        return final_df

    except Exception as e:
        return pd.DataFrame({"Error": [f"Ranking Error: {str(e)}"]})

In [33]:
from deep_translator import GoogleTranslator
def predict_sentiment_auto(feedback_text,dept_name, subject_name):
    if not feedback_text or not feedback_text.strip():
        return "ERROR", "No Text Provided"

    try:
        feedback_text = GoogleTranslator(source='auto', target='en').translate(feedback_text)
    except Exception as e:
        feedback_text = feedback_text

    context = "General/Other"
    t = feedback_text.lower()
    keywords = {
        "Instructor Performance": ["teacher", "prof", "teaching", "instructor", "lecturer"],
        "Assessment & Grading": ["exam", "test", "quiz", "grade", "score", "assignment"],
        "Course Curriculum": ["course", "content", "material", "syllabus", "module"],
        "Learning Facilities": ["lab", "room", "wifi", "facility", "projector", "canteen"]
    }

    for category, words in keywords.items():
        if any(re.search(rf"\b{w}", t) for w in words):
            context = category
            break

    clean = preprocess(feedback_text)
    X_text_feat = vectorizer.transform([clean])

    sarcasm_flag = detect_sarcasm(feedback_text)
    X_sarcasm_feat = np.array([[sarcasm_flag]])
    sarcasm_msg = f"နောက်ပြောင်ပြောတဲ့စာ ပါဝင်နိုင်ခြေ: {'ရှိနိုင်သည်' if sarcasm_flag == 1 else 'မရှိပါ'}"

    X_combined = hstack([X_text_feat, X_sarcasm_feat])

    probs = sentiment_model.predict_proba(X_combined)

    if sarcasm_flag == 1:
        probs[0][2] *= 0.5

    pred_index = np.argmax(probs)
    sentiment_pred = sentiment_le.inverse_transform([pred_index])[0]

    confidence_percent = round(probs[0][pred_index] * 100, 2)


    sentiment_label_mm = {
            "POSITIVE": "အကောင်းမြင် (Positive)",
            "NEGATIVE": "အဆိုးမြင် (Negative)",
            "NEUTRAL": "ကြားနေ (Neutral)"
        }

    sentiment_translated = sentiment_label_mm.get(sentiment_pred.upper(), sentiment_pred)
    final_sentiment_msg = f"စိတ်ခံစားမှု ခန့်မှန်းချက်: {sentiment_translated}"

    result_text = f"{final_sentiment_msg} \n ({sarcasm_msg})"




    prediction_history.append({
        "Department": dept_name,
        "Subject": subject_name,
        "Sentiment": sentiment_pred.upper(),
        "Context": context,
    })

    ranking_df = get_course_ranking(prediction_history)

    return result_text,context

In [46]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def student_analytics(selected_department):
    df_combined = pd.DataFrame(prediction_history)
    df_combined['Sentiment'] = df_combined['Sentiment'].str.upper().str.strip()

    if selected_department != "အားလုံး":
        df_filtered = df_combined[df_combined['Department'] == selected_department]
    else:
        df_filtered = df_combined.copy()


    counts = df_filtered.groupby(['Department', 'Sentiment']).size().unstack(fill_value=0).reset_index()


    fig1, ax1 = plt.subplots(figsize=(8, 5))


    modern_colors = ['#fb7185', '#94a3b8', '#10b981']

    counts.set_index('Department').plot(
        kind='bar',
        stacked=True,
        ax=ax1,
        color=modern_colors,
        edgecolor='white',
        linewidth=1
    )

    ax1.set_title("Unified Sentiment Distribution", fontsize=12, fontweight='bold', pad=15)
    ax1.legend(title="Sentiment", labels=['Negative', 'Neutral', 'Positive'])
    plt.tight_layout()


    sentiment_map = {"POSITIVE": 1, "NEUTRAL": 0, "NEGATIVE": -1}
    df_filtered['score'] = df_filtered['Sentiment'].map(sentiment_map)
    pivot = df_filtered.pivot_table(index='Subject', columns='Department', values='score', aggfunc='mean')

    fig2, ax2 = plt.subplots(figsize=(10, 6))
    sns.heatmap(pivot, annot=True, cmap='RdYlGn', center=0, ax=ax2, linewidths=0.5)
    plt.tight_layout()

    return counts, fig1, fig2

In [47]:
import gradio as gr

custom_theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="blue",
    neutral_hue="slate",
).set(
    button_primary_background_fill="*primary_500",
    button_primary_background_fill_hover="*primary_600",
    block_title_text_weight="700",
    block_shadow="*shadow_drop_lg"
)

dept_list = sorted(data_set['department'].unique().tolist())
dept_subject_map = {dept: sorted(data_set[data_set['department'] == dept]['subject_name'].unique().tolist()) for dept in dept_list}

def update_subjects(dept):
    return gr.update(choices=dept_subject_map.get(dept, []), value=dept_subject_map.get(dept, [])[0] if dept_subject_map.get(dept) else None)

custom_css = """
.container { max-width: 800px; margin: auto; }
h1 { text-align: center; color: #4F46E5; font-family: 'Helvetica', sans-serif; }
.description { text-align: center; font-size: 1.1em; color: #475569; }
"""

with gr.Blocks(theme=custom_theme, css=custom_css, title="Automated Analysis Of Course Evalution Comments") as demo:
    with gr.Column(elem_classes=["container"]):
        gr.Markdown("# 🤖 Automated Analysis Of Course Evalution Comments")
        gr.Markdown("*ကျောင်းသားများ၏ Feedback ကို AI အသုံးပြု၍ အလိုအလျောက် စစ်ဆေးပေးသောစနစ်*", elem_classes=["description"])

    with gr.Tabs():

        with gr.TabItem("🔍 ခန့်မှန်းရန် (Prediction)"):
            with gr.Row():
                with gr.Column():

                    dept_select = gr.Dropdown(
                        choices=dept_list,
                        label="ဌာနရွေးချယ်ရန် (Select Department)",
                        interactive=True
                    )
                    subject_select = gr.Dropdown(
                        choices=[],
                        label="ဘာသာရပ်ရွေးချယ်ရန် (Select Subject)",
                        interactive=True
                    )

                    input_text = gr.Textbox(
                        lines=4,
                        placeholder="ဒီနေရာမှာ ရေးပါ... (ဥပမာ - စာသင်တာ ကောင်းပေမဲ့ အခန်းက အရမ်းပူတယ်)",
                        label="အကြံပြုချက် (Input Feedback)",
                        show_label=True
                    )

                    submit_btn = gr.Button("🚀 ရလဒ်စစ်ဆေးမည်", variant="primary", scale=1)

                with gr.Column():
                    output_text = gr.Textbox(label="စစ်ဆေးတွေ့ရှိချက် (Result)", lines=4, interactive=False)
                    output_context = gr.Textbox(label="သက်ဆိုင်သည့်အကြောင်းအရာ (Category)", interactive=False)




            dept_select.change(fn=update_subjects, inputs=dept_select, outputs=subject_select)

            gr.Examples(
                examples=[
                    ["The teacher is great but the exam was too hard."],
                    ["ဆရာက မေးခွန်းတိုင်းကို သေချာ ဖြေကြားပေးပြီး students တွေကို လေ့ကျင့်ခန်းအတွက် လိုအပ်တဲ့ guidance ကိုပေးတယ်။"],
                    ["Projector and lab equipment do not work properly, making the classroom experience terrible."],
                    ["Perfect, the wifi never works when I need it the most."],
                    ["The lecture is too long and exhausting."]
                ],
                inputs=input_text
            )


            submit_btn.click(
                fn=predict_sentiment_auto,
                inputs=[input_text, dept_select, subject_select],
                outputs=[output_text, output_context]
            )

        with gr.TabItem("📊 Course Ranking Dashboard"):
              selector = gr.Radio(
                  choices=["Department", "Subject"],
                  value="Subject",
                  label="Ranking တွက်ချက်လိုသော ကဏ္ဍကို ရွေးပါ"
              )

              output_table = gr.Dataframe(interactive=False)


              selector.change(
                  fn=lambda key: get_course_ranking(prediction_history, key),
                  inputs=selector,
                  outputs=output_table
              )

              demo.load(
                  fn=lambda key: get_course_ranking(prediction_history, key),
                  inputs=selector,
                  outputs=output_table
              )

        with gr.TabItem("Student Pulse Dashboard"):
            gr.Markdown("# 📊 Student Feedback Summary Analytics")
            dept_input = gr.Dropdown(choices=["အားလုံး"] + sorted(data_set['department'].unique().tolist()),label="ဌာနရွေးချယ်ရန်", value="အားလုံး")

            with gr.Tabs():
              with gr.TabItem("📈 အရေအတွက် တွက်ချက်မှု"):
                output_df = gr.Dataframe(label="Sentiment Counts Table")
                plot_count = gr.Plot(label="Stacked Bar Chart")

              with gr.TabItem("🗺️ အသေးစိတ် Heatmap"):
                plot_heatmap = gr.Plot(label="Subject-wise Analysis")

            dept_input.change(fn=student_analytics, inputs=dept_input, outputs=[output_df, plot_count, plot_heatmap])



        with gr.TabItem("ℹ️ အလုပ်လုပ်ပုံ (About)"):
            gr.Markdown("""
            ### 🧠 ဤစနစ် ဘယ်လိုအလုပ်လုပ်သလဲ?

            ဤစနစ်သည် **Logistic Regression Classifier** ကို အသုံးပြုပြီး အောက်ပါ အချက် ၃ ချက်ကို အဓိကထား စစ်ဆေးပေးပါတယ် -

            1.  **Sentiment Analysis:** စာသားက Positive (အကောင်း)၊ Negative (အဆိုး) သို့မဟုတ် Neutral (ကြားနေ) ဖြစ်မဖြစ် ခွဲခြားခြင်း။

            2.  **Sarcasm Detection:** စာသားက အပေါ်ယံကောင်းသယောင် ရှိပေမဲ့ "သရော်စာ/နောက်ပြောင်ပြောတဲ့စာ (Sarcasm)" ပါဝင်နေခြင်း ရှိမရှိ စစ်ဆေးခြင်း။

            3.  **Project Context & Feedback Classification:** စာသား အတွင်း keyword အခြေခံပြီး အလိုအလျောက် **feedback context/topic** ကို ခွဲခြားခြင်း။

            4.  **Course Ranking:** ကျောင်းသားတွေရဲ့ စစ်မှန်တဲ့ တုံ့ပြန်မှုတွေအပေါ် အခြေခံပြီး အရည်အသွေးအကောင်းဆုံး ဘာသာရပ်တွေကို Ranking ထုတ်ပေးရန်။

            📊 ထို့နောက် ရလဒ်များကို Dashboard အနေဖြင့် ပြသပေးပြီး ဌာန အခြေအနေများ၊ သင်ကြားမှု အရည်အသွေးနှင့် ကျောင်းသားများ၏ “Student Pulse” ကို အလွယ်တကူ သိရှိနိုင်စေပါသည်။

            #### 🛠 အသုံးပြုထားသော Libraries များ
            * **Scikit-Learn:** Sentiment ခန့်မှန်းပေးမည့် **Logistic Regression** မော်ဒယ်လ်အတွက်။
            * **NLTK:** စကားလုံးများ ခွဲခြားသန့်စင်ရန်။
            * **Deep-Translator:** မြန်မာစာကို Google Translate ဖြင့် English သို့ ပြောင်းလဲရန်။
            * **TextBlob:** Sarcasm စစ်ဆေးရာတွင် စိတ်ခံစားမှုပြင်းအား (Polarity) တိုင်းတာရန်။
            * **Matplotlib & Seaborn:** ဒေတာများကို **Heatmap** နှင့် ဇယားများဖြင့် Visualise ပြုလုပ်ရန်။
            * **Gradio:** ခေတ်မီသော User Interface ဖန်တီးရန်။


            > **မှတ်ချက်:** Data နည်းပါးသေးသည့်အတွက် ရလဒ်များ တစ်ခါတရံ လွဲချော်နိုင်ပါသည်။
            """)


    submit_btn.click(
        fn=predict_sentiment_auto,
        inputs=input_text,
        outputs=[output_text, output_context]
    )

demo.launch()

/tmp/ipykernel_472/3691873129.py:26: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme, css=custom_css, title="Automated Analysis Of Course Evalution Comments") as demo:
/tmp/ipykernel_472/3691873129.py:26: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme, css=custom_css, title="Automated Analysis Of Course Evalution Comments") as demo:
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1052: UserWarning: Expected 3 arguments for function <function predict_sentiment_auto at 0x7b673c329ee0>, received 1.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1056: UserWarning: Expected at least 3 arguments for function <function predict_sentiment_auto at 0x7b673c329ee0>, received 1.
  wa

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f5014eda0ef701128f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
